# 1장. Agents API 핵심 개념

공통 sales.csv의 입력과 반환할 JSON 구조를 살펴보고, Agent·Session·Turn·Environment·Harness를 실제 요청 구성과 연결합니다. 이 장은 API 키·모델 설정 없이 로컬에서 실행하며 API 접근 권한이나 원격 분석 성공을 검증하지 않습니다.


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · Colab 업로드용


## 학습 안내

위에서 아래로 실행하며 입력 구조 → 반환 형식 → 구성 요소 → 요청 미리보기 → 실행 환경을 확인합니다. CSV 집계 정답을 계산하거나 Session을 생성하지 않습니다.

| 학습 단계 | 이 장의 역할 | 다음 실습 |
| --- | --- | --- |
| 빠른 시작 | 작은 작업으로 생성·스트리밍·정리 경험 | [빠른 시작](../quickstart.ipynb) |
| 1장 | 데이터·핵심 용어·요청과 결과 구조 이해 | 현재 노트북 |
| 2장 | CSV를 메시지로 보내고 텍스트 JSON 응답 확인 | [첫 CSV 분석](02_first_agent.ipynb) |
| 5장 | 입력 파일과 결과 Artifact 다운로드 | [파일과 산출물](05_files_artifacts.ipynb) |

완료하면 모델·지침을 어디에 넣는지, 같은 Session에서 다음 Turn을 요청한다는 의미, Python 실행 위치, JSON 응답과 결과 파일의 차이를 설명할 수 있습니다.

각 장은 같은 데이터를 사용하는 독립 기능 실습입니다. 전체 학습 경로와 장별 입출력 변화는 [공통 학습·실습 안내](../../docs/validation.md)에서 확인하세요.


**실습 범위:** 모든 코드는 로컬 입력 확인과 요청 데이터 미리보기입니다. API 생성·이벤트 처리·원격 결과 확인·자원 정리는 2장에서 실습합니다. 별도 검증 도구의 오프라인 통과는 API 실행 성공을 뜻하지 않습니다. [실습 안내](../../docs/validation.md)


## 0. 로컬 실습 준비

로컬은 저장소 루트에서 uv sync --locked, uv run jupyter lab을 실행하고 .venv 커널을 선택하세요. Colab은 다운로드 안내의 독립 실행본을 업로드하고 포함 파일 준비 셀까지 실행합니다. 이 장에서는 API 키나 모델을 설정할 필요가 없습니다.

설치·커널 문제는 [환경 준비](../setup.ipynb)를 참고하세요. 아래 코드는 저장소 또는 Colab 임시 폴더에서 공통 CSV를 읽을 뿐 API client를 만들지 않습니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


In [ ]:
import csv
import io
import json
from tutorial_helpers.common import project_root

root = project_root()
csv_path = root / "data/sales.csv"
csv_text = csv_path.read_text(encoding="utf-8")
print("Local input:", csv_path)


## 1. 입력 CSV 구조 확인

각 행은 날짜·상품·수량·단가를 가진 거래입니다. 행별 매출 규칙은 quantity × unit_price입니다. 날짜와 상품은 분류 기준, 수량과 단가는 계산 입력입니다. 금액은 실습용 정수이며 입력에 없는 세금·환불·통화 변환은 추가하지 않습니다.

아래에서 CSV 원문·열 이름·첫 행을 확인합니다. 아직 합계나 상품별 집계는 계산하지 않습니다. 첫 행의 값이 어떤 의미인지 설명한 뒤 다음 절의 반환 형식으로 넘어가세요.


In [ ]:
print(csv_text)
reader = csv.DictReader(io.StringIO(csv_text))
print("Columns:", reader.fieldnames)
print("First transaction:", next(reader))


## 2. 반환할 JSON 구조와 파일 구분

결과 계약은 계산 정답이 아니라 필드·자료형·의미를 정합니다. 2장에서는 모델의 최종 텍스트를 JSON으로 읽습니다. JSON으로 해석할 수 있다는 사실만으로 계산이 정확하거나 Python이 실행됐다고 단정하지 않습니다.

| 필드 | 의미 | 자료형 |
| --- | --- | --- |
| row_count | 거래 행 수 | 정수 |
| total_quantity | 판매 수량 합계 | 정수 |
| total_revenue | 각 행의 수량 × 단가 합계 | 정수 |
| by_product | 상품 이름을 키로 갖는 상품별 매출 | 객체: 상품 → 정수 |

5장에서는 summary.json과 report.md를 sandbox에서 만들고 Artifact로 내려받습니다. 같은 숫자를 JSON과 Markdown으로 표현할 수 있지만, 응답 텍스트·sandbox 파일·로컬 다운로드는 서로 다른 대상입니다. 이 장에서는 파일을 만들지 않습니다. CSV 기준 정답과 상세 내용 검사는 별도 검증 도구에서 수행합니다.


## 3. Agent · Session · Turn · Environment · Harness

| 구성 요소 | 역할 | 이 예제에서의 의미 |
| --- | --- | --- |
| Agent 설정 | 모델·지침·도구 등 동작 지정 | Python으로 계산하는 매출 분석 담당 지침 |
| Session | 대화와 작업을 이어가는 상태 단위 | 같은 CSV 분석을 이어가는 대화 |
| Turn | 입력을 받아 수행하는 작업 단위 | 전체 매출 분석 후, 다음 Turn에서 일별 분석 요청 |
| Environment | 명령·코드·파일 작업의 실행 환경 | CSV를 읽고 Python을 실행하는 sandbox |
| Harness | 모델 호출과 도구 실행의 루프 조율 | 계산·명령·응답의 다음 행동 결정 |

Agent 설정을 Session 생성 요청에 직접 넣을 수 있으며, 별도 저장 Agent 리소스를 먼저 만들 필요는 없습니다. 저장 Agent 재사용은 [4장](04_agent_configuration.ipynb)에서 실습합니다. 같은 Session에서 다음 작업을 요청하면 별도 Turn이 생기며, 이는 새 Session을 만드는 것과 다릅니다.

Agents API에서는 OpenAI가 Harness를 운영합니다. environment.type은 명령을 실행할 환경의 선택입니다. 애플리케이션의 외부 함수 실행·업무 권한·사람 승인까지 자동으로 대신한다는 뜻은 아닙니다.


### 요청이 실행되는 흐름

애플리케이션이 작업을 제출하면 Session의 Turn이 시작되고, Harness가 모델·도구 루프를 진행합니다. 코드·파일이 필요하면 Environment에서 명령을 실행하고 애플리케이션은 이벤트와 결과를 받습니다.

```mermaid
flowchart LR
    A[Application] -->|agent + environment + input| H[Agents API Harness]
    H -->|commands| E[Environment]
    E -->|command results| H
    H -->|events and final result| A
```

이 그림은 역할 관계를 설명하며 HTTP 호출 순서나 전체 API 스키마가 아닙니다. 2장에서는 애플리케이션 역할을 노트북 Python 코드가 수행합니다. 공식 근거: [Architecture](https://developers.openai.com/api/docs/guides/agents-api/architecture).


## 4. 요청 구성 미리보기

다음은 2장에서 사용할 agent·environment·input·stream을 연결한 요청 데이터입니다. 모델 ID는 설명용 자리표시자이며, 이 셀은 client나 Session을 만들거나 API에 전송하지 않습니다. 모델 선택·인증은 2장의 준비 단계에서 설정합니다.

| 개념 | 요청의 위치 |
| --- | --- |
| 모델·지침 | agent.model, agent.instructions |
| Python 실행 위치 | environment.type |
| 요청할 작업과 CSV | input |
| 이벤트로 진행 관찰 | stream |

stream=True는 진행 관찰 방식이며 작업 성공·올바른 계산·파일 생성을 보장하는 설정이 아닙니다. 2장에서 실제 생성 호출과 완료 상태 확인을 연결합니다.


In [ ]:
request_preview = {
    "agent": {
        "model": "MODEL_ID_FOR_CHAPTER_2",
        "instructions": "Use Python to calculate sales metrics from the provided CSV.",
    },
    "environment": {"type": "openai_hosted"},
    "input": (
        "Calculate row_count, total_quantity, total_revenue, and by_product. "
        "Return only a JSON object.\nCSV input:\n" + csv_text
    ),
    "stream": True,
}
print(json.dumps(request_preview, indent=2))


## 5. 실행 환경 선택

아래는 환경 유형을 비교하는 설정 조각이며 완성된 Session 생성·연결 코드가 아닙니다.

| 환경 | 선택 기준 | 이후 실습 |
| --- | --- | --- |
| none | 자체 sandbox 명령·파일 없이 응답하거나 외부 도구 사용 | 개념 비교 |
| openai_hosted | OpenAI가 관리하는 sandbox에서 Python·파일 작업 | 2장 이후 기본 경로 |
| self_hosted | 사내 네트워크·패키지가 있는 실행 환경 연결 | 12장 |

none에서도 외부 도구를 사용할 수 있지만 자체 sandbox의 Bash·workspace 파일은 제공되지 않습니다. self_hosted는 실행 환경을 직접 연결하며 Harness를 직접 운영한다는 뜻이 아닙니다. 필요한 연결·수명주기 관리는 [12장](12_self_hosted.ipynb)을 참고하세요.

[15장 Multi-agent](15_multi_agent.ipynb)의 하위 에이전트도 같은 환경의 파일시스템을 공유합니다. 하위 에이전트 생성이 별도 sandbox 생성을 의미하지는 않습니다.


In [ ]:
environment_examples = {
    "No sandbox commands or workspace files": {"type": "none"},
    "Python analysis in an OpenAI-managed sandbox": {"type": "openai_hosted"},
    "Execution on private infrastructure": {"type": "self_hosted"},
}
print(json.dumps(environment_examples, indent=2))


## 6. Responses API · Agents SDK · Agents API 구분

| 선택 | 애플리케이션이 주로 관리할 부분 | Python 진입점 |
| --- | --- | --- |
| Responses API | 요청·대화 연결·도구 실행 흐름 구성 | client.responses |
| Agents SDK | 애플리케이션에서 실행하는 에이전트 루프 | Agent, Runner |
| Agents API | 관리되는 Session에 작업 제출·이벤트·결과 처리 | client.beta.agents |

이 교재는 OpenAI Python SDK의 client.beta.agents를 사용합니다. from agents import Agent, Runner는 별도 Agents SDK의 진입점입니다. SDK Guardrail 데코레이터를 Agents API 필드로 옮기지 않습니다. 업무 검증·승인 흐름은 11장에서 배웁니다.

Codex App Server 직접 운영은 이 입문 과정의 필수 조건이 아닙니다. 먼저 이 교재의 API 호출 표면과 애플리케이션·Harness·환경의 책임을 구분하세요.


## 자기 점검과 다음 장

다음 질문에 답한 뒤 [2장 CSV 분석 에이전트 실행](02_first_agent.ipynb)으로 이동하세요.

| 질문 | 확인할 개념 |
| --- | --- |
| 모델과 분석 지침은 어디에 넣나요? | agent.model·agent.instructions |
| 같은 Session에서 다음 분석을 요청하면 무엇이 생기나요? | 별도 Turn이며 새 Session과 구분 |
| CSV를 읽는 Python 명령은 어디에서 실행되나요? | 이 예제에서는 openai_hosted Environment |
| stream=True이면 계산 정답이 보장되나요? | 진행 관찰과 완료·정답 확인은 별개 |
| 응답 JSON과 summary.json 다운로드는 같은가요? | 응답 텍스트·sandbox 파일·로컬 파일은 별개 |

이 장의 완료 기준은 CSV 구조·반환 필드·요청 위치·실행 주체를 설명할 수 있는 것입니다. 원격 자원을 만들지 않아 Session 정리도 필요하지 않습니다. 로컬 검사 통과를 API 실행 검증으로 해석하지 않습니다.


## 공식 참고 자료

### Agents API 전용 문서

- [Agents API overview](https://developers.openai.com/api/docs/guides/agents-api/overview) — Agents API의 개념과 관리형 Harness 역할.
- [Architecture](https://developers.openai.com/api/docs/guides/agents-api/architecture) — Harness·애플리케이션·실행 환경의 책임 구분.

### 보조 자료 — 공통 문서

- [Agents — runtime비교](https://developers.openai.com/api/docs/guides/agents#compare-agent-runtime-options) — Agents API·Agents SDK 등 여러 실행 방식을 비교하는 공통 안내입니다.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
